# Football Managerial Decision Analytics — Chapter 01
## Data source, provenance, quality audit and exploratory analysis

> **Data.** StatsBomb Open Data (https://github.com/hudl/open-data), provided by StatsBomb / Hudl, used for educational, non-commercial research. Raw data are downloaded from the official repository at run time and are not redistributed here. Every figure carries its chart ID from `docs/VISUALIZATION_CATALOGUE.md`.

This notebook is one chapter of the research record. All logic lives in the shared `src/` package; the cells call it. The same charts are available interactively in the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
# Setup — locate (or fetch) the project code, import the shared package, load the workspace
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/blazingarrows1525/football-managerial-decision-analytics"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "workspace.py").exists()), None)
if ROOT is None:  # opened on its own (e.g. Google Colab): fetch the project code once, next to this notebook
    ROOT = Path.cwd() / "football-managerial-decision-analytics"
    if not (ROOT / "src" / "workspace.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FMA_PROJECT_ROOT", str(ROOT))  # data cache and outputs live inside the project folder

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.reporting.notebook_tools import (catalogue, explorer, implication, interpret, limitation, show, start,
                                          timed, timings)

pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)
RUN_ADVANCED_GALLERY = True   # heavier, optional sections; the core story does not depend on them
SEED = 42                             # every resampling and model step takes this seed explicitly

ws = start(seed=SEED)                 # first run downloads and parses; later runs read the processed cache
B, T, FOCUS = ws.bundle, ws.tables, ws.focus
tm = ws.team_matches()                # the focus team's matches in date order
print(f"Advanced sections: {'on' if RUN_ADVANCED_GALLERY else 'off'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## B · Data source, provenance and quality

**Research question.** Where do the data come from, under which terms, and can the parsed tables be trusted
for the analyses built on them?

**Data and method.** The download client records the URL, byte size, SHA-256 hash and write time of every raw
file. Parsed tables are checked against sixteen domain rules and fourteen table contracts (grain, key, required
fields).

In [ ]:
from src.data import provenance
from src.data.schemas import check_contracts

man = B.manifest
display(provenance.source_register()[["source", "role_in_project", "terms", "status"]])
display(man.groupby(["resource", "status"]).agg(files=("relative_path", "size"), MB=("bytes", lambda s: round(s.sum() / 1e6, 2))))
display(man[["relative_path", "bytes", "sha256", "file_written_utc"]].head(4))
print("Terms recorded from the source licence:")
for t in provenance.TERMS_SUMMARY:
    print(" -", t)
interpret(f"{len(man)} raw files ({man.bytes.sum() / 1e6:.1f} MB) were read with {int((man.status != 'ok').sum())} failures; "
          f"each has a SHA-256 hash in `data/manifest/source_manifest.json`. The user agreement allows research "
          f"use and public sharing of analysis but not redistribution of the data, so this repository contains "
          f"no raw or processed event tables.")

In [ ]:
show("A-001", ws)
contracts = check_contracts(B)
display(contracts[["table", "origin", "rows", "grain", "status", "duplicate_keys"]])
show("A-016", ws)
q = B.quality
interpret(f"{int((q.status == 'PASS').sum())} of {len(q)} domain rules and {int((contracts.status == 'PASS').sum())} of "
          f"{len(contracts)} table contracts pass: event IDs are unique, coordinates lie on the 120 × 80 pitch, xG is "
          f"within [0, 1] and attached only to shots, every team-match has eleven starters and one goalkeeper, "
          f"on-pitch spells do not overlap and the goals rebuilt from events equal the published scores.")
limitation("Passing validation shows internal consistency, not that the provider's event tagging is error-free.")

### Missing values, duplicates and outliers

**Question.** Which gaps are structural, and are extreme values errors or football?

In [ ]:
show("A-003", ws)
show("A-005", ws)
ev = T["events"]
shots = ev[ev.type == "Shot"]
interpret(f"Most fields are sparse by design: `shot_xg` is present on {ev.shot_xg.notna().mean():.1%} of all events but on "
          f"{shots.shot_xg.notna().mean():.0%} of the {len(shots):,} shots; pass end locations exist for "
          f"{ev.loc[ev.type == 'Pass', 'end_x'].notna().mean():.0%} of passes. Missing values are kept as missing and "
          f"are never read as zero.")

In [ ]:
from src.visualization import audit
dups = audit.duplicate_audit_table(B)
show("A-006", ws)
show("A-010", ws)
interpret(f"{int(dups.full_row_duplicates.sum())} duplicated rows and {int(dups.key_duplicates.fillna(0).sum())} duplicated keys across "
          f"{len(dups)} tables. Values beyond 1.5 × IQR are real events (long passes, penalties, long stoppages) and are kept.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("A-004", ws)
    show("A-008", ws)
    show("A-009", ws)

### What the data support, and what they do not

**Question.** Which analytical inputs exist? The answer is read from the tables, not assumed.

In [ ]:
show("A-012", ws)
show("A-011", ws)
fa = audit.feature_availability_table(B)
absent = fa[fa.status != "available"].feature.tolist()
interpret(f"{int((fa.status == 'available').sum())} of {len(fa)} input groups are present. Not available: "
          + "; ".join(absent) + ". None of these is imputed, and no chart that needs them is drawn.")
implication("Selection and substitution tools built on this data cannot know who is injured or tired; the analyst "
            "must supply availability.")

In [ ]:
# Optional 360 freeze frames: published by the source, downloaded per match only on request.
# When files are cached, check that their identifiers actually join to the event file before using them.
import json
d360 = B.config.raw_dir / "three-sixty"
rows = []
for p in sorted(d360.glob("*.json")) if d360.exists() else []:
    frames = json.loads(p.read_text(encoding="utf-8"))
    ids = set(ev.loc[ev.match_id == int(p.stem), "event_id"])
    rows.append({"match_id": int(p.stem), "frames": len(frames), "joined_to_events": len(ids & {f["event_uuid"] for f in frames})})
check360 = pd.DataFrame(rows)
if len(check360):
    display(check360)
    bad = check360[check360.joined_to_events == 0]
    interpret(f"{len(check360)} optional 360 file(s) are cached. " + (f"{len(bad)} of them share no identifier with their "
              f"event file — a source inconsistency; those frames are not used." if len(bad) else "All of them join to their events."))
else:
    print("No 360 file is cached; chart E-129 is reported as implemented_but_unavailable.")
show("A-014", ws)

## C · Feature engineering and exploratory analysis

**Question.** After normalising for playing time and role, how are player metrics distributed and how do they
relate?

**Data and method.** Minutes are rebuilt from the event stream. Per-90 values are `value / minutes × 90` and
percentiles are computed within the player's role group among players above the minutes threshold.

In [ ]:
from src.features.player_features import metric_dictionary
from src.reporting.report_builder import DERIVED_METRICS
from src.workspace import EDA_METRICS

display(metric_dictionary().head(12))
display(pd.DataFrame(DERIVED_METRICS, columns=["metric", "formula", "inputs", "caveat"]).head(8))
s = B.season
print(f"{int(s.eligible.sum())} of {len(s)} player-team rows have at least {B.config.min_minutes:.0f} minutes; "
      f"{int((s.minutes_played >= 900).sum())} have at least 900.")
show("L-269", ws)

In [ ]:
from src.analytics import descriptive as desc
rm = desc.role_medians(B, ["xg_p90", "key_passes_p90", "progressive_passes_p90", "interceptions_p90", "pass_completion_pct"])
display(rm)
show("G-156", ws)
show("B-018", ws, metric="xg_p90")
fams = rm.drop(index="Goalkeeper", errors="ignore")
interpret(f"Role families separate clearly on some metrics: median xG/90 is highest for {fams.xg_p90.idxmax()}s "
          f"({fams.xg_p90.max():.2f}) and lowest for {fams.xg_p90.idxmin()}s ({fams.xg_p90.min():.2f}); interceptions/90 "
          f"peak for {fams.interceptions_p90.idxmax()}s ({fams.interceptions_p90.max():.2f}). Group sizes: "
          + ", ".join(f"{i} {int(n)}" for i, n in rm.players.items()) + ".")
implication("Compare players inside their role group. A raw leaderboard mostly re-discovers positions.")

In [ ]:
from src.analytics import statistical_tests as stt
el = ws.eligible()
r, n = stt.correlation_matrix(el, EDA_METRICS, "spearman")
show("J-200", ws)
pairs = r.where(np.triu(np.ones(r.shape, bool), 1)).stack().sort_values()
lo, hi = pairs.index[0], pairs.index[-1]
interpret(f"Among {len(el)} eligible players the strongest positive rank correlation is between {hi[0]} and {hi[1]} "
          f"(ρ = {pairs.iloc[-1]:.2f}) and the strongest negative between {lo[0]} and {lo[1]} (ρ = {pairs.iloc[0]:.2f}). "
          f"Pairwise n ranges from {int(n.values.min())} to {int(n.values.max())}.")
limitation("Players are nested in teams and opponents are observed only against the focus team; these are "
           "descriptive associations.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("B-019", ws, metric="progressive_passes_p90")
    show("B-022", ws, metric="pressures_p90")
    show("D-074", ws)
    show("L-246", ws)

In [ ]:
display(timings().tail(12))